In [ ]:
!pip install pytrends requests pandas numpy matplotlib google-play-scraper
!pip install "urllib3<2"

In [ ]:
!mkdir -p manual_exports

In [ ]:
!python 01_extract_google_trends.py

Processing: data_milestones_manual.csv...
Saved dataset: ./processed_data/clean_data_milestones_manual.csv
Saved plot: ./processed_data/plot_data_milestones_manual.png


In [ ]:
!python 01b_parse_manual_trends_export.py

Parsing ./manual_exports/multiTimeline_2016.csv.csv...
  Found columns: ['date', 'Google Translate', 'ChatGPT', 'DeepL'], 127 rows
Parsing ./manual_exports/multiTimeline_5_years.csv...
  Found columns: ['date', 'Google Переводчик', 'ChatGPT', 'DeepL'], 262 rows

Saved combined file to ./data/google_trends_raw.csv (380 rows, columns: ['date', 'Google Translate', 'ChatGPT_x', 'DeepL_x', 'Google Переводчик', 'ChatGPT_y', 'DeepL_y'])
This file has the same schema 04_merge_and_process.py expects -- you can now run that script normally.


In [ ]:
!python 02_extract_wikipedia_pageviews.py

Fetching pageviews for Google_Translate...
  Got 133 monthly records
Fetching pageviews for DeepL...
  Got 102 monthly records
Fetching pageviews for ChatGPT...
  Got 44 monthly records
Fetching pageviews for Neural_machine_translation...
  Got 131 monthly records
Saved 410 rows to ./data/wikipedia_pageviews_raw.csv


In [ ]:
!python 03_extract_app_reviews.py

Fetching reviews for google_translate (com.google.android.apps.translate)...
  Got 2000 reviews
Fetching reviews for deepl (com.deepl.mobiletranslator)...
  Got 2000 reviews
Fetching reviews for chatgpt (com.openai.chatgpt)...
  Got 2000 reviews
/content/03_extract_app_reviews.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_frames, ignore_index=True)
Saved 6000 rows to ./data/app_reviews_raw.csv


In [ ]:
import pandas as pd

file_path = "./data/google_trends_raw.csv"

# Read raw trends data
df = pd.read_csv(file_path)

# Convert '<1' to 0.5 and ensure numeric data types
for col in df.columns:
    if col != "date":
        df[col] = df[col].astype(str).str.replace("<1", "0.5")
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Save cleaned CSV back to the same location
df.to_csv(file_path, index=False)
print(
    "Data successfully cleaned: '<1' values converted to numeric in google_trends_raw.csv"
)

Data successfully cleaned: '<1' values converted to numeric in google_trends_raw.csv


In [ ]:
import pandas as pd

raw_path = "./data/google_trends_raw.csv"
df = pd.read_csv(raw_path)

# Rename translated entity names to unified English names
rename_map = {
    "Google Переводчик": "Google Translate",
    "Google Переводчик_x": "Google Translate",
    "Google Переводчик_y": "Google Translate",
    "Google Translate_x": "Google Translate",
    "Google Translate_y": "Google Translate",
    "ChatGPT_x": "ChatGPT",
    "ChatGPT_y": "ChatGPT",
    "DeepL_x": "DeepL",
    "DeepL_y": "DeepL",
}

df = df.rename(columns=rename_map)

# Group duplicate columns by name and take their max/mean value
df = df.groupby(level=0, axis=1).first()

# If duplicate columns remain due to merging, combine them
if df.columns.duplicated().any():
    df = df.groupby(lambda x: x, axis=1).mean()

# Clean numeric values
for col in df.columns:
    if col != "date":
        df[col] = df[col].astype(str).str.replace("<1", "0.5")
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df.to_csv(raw_path, index=False)
print("Updated ./data/google_trends_raw.csv with clean, unified columns.")

Updated ./data/google_trends_raw.csv with clean, unified columns.


/tmp/ipykernel_2145/3768436693.py:22: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df = df.groupby(level=0, axis=1).first()


In [ ]:
!python 04_merge_and_process.py

Master dataset saved: ./data/master_dataset.csv (664 rows)


In [ ]:
!python 05_analyze.py


--- Segmented trend: Google Translate, pre vs post ChatGPT ---
{'pre_chatgpt': {'slope': np.float64(-0.10501953613913642), 'intercept': np.float64(21.844754628269175), 'n_points': 82}, 'post_chatgpt': {'slope': np.float64(-0.07513614404918754), 'intercept': np.float64(11.788603425559952), 'n_points': 45}}

--- Cross-correlation: Google Translate vs ChatGPT (Trends index) ---
    lag_months  correlation
0           -6    -0.647473
1           -5    -0.640403
2           -4    -0.634745
3           -3    -0.629858
4           -2    -0.624800
5           -1    -0.618788
6            0    -0.612785
7            1    -0.602951
8            2    -0.591438
9            3    -0.578580
10           4    -0.566819
11           5    -0.555247
12           6    -0.548179

--- Relative share of attention across entities ---
entity       ChatGPT     DeepL  Google Translate
month                                           
2026-03-01  0.857143  0.017857          0.125000
2026-04-01  0.861111  0.01851